# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shakhaoathpappu-jpg/FlyRank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule: Content whose search data hasn't been updated recently (staleness) and whose CTR is meaningfully below what similarly-ranked pages get (CTR gap) is prioritized for refresh.

Signal 1 — Staleness (behind refresh flags): MIXED. Decline rate rises from 0-30d (12.8%) to 31-90d (20.3%), which supports the rule — but then drops at 91-180d (5.0%) and 181-365d (0.9%). This isn't monotonic, and the tail buckets have very small n (1324 and 228 rows vs 149,439 in the 0-30d bucket), so I'm treating this as a partial, not clean, confirmation — the signal works in the near term but the long-stale bucket may be biased (only content that survived without being touched, e.g. evergreen pages that never needed updates).

Signal 2 — CTR vs position (behind CTR-fix logic): [fill in after rerunning — see below]

Reason codes this rule can output: STALE, CTR_GAP, STALE_AND_CTR_GAP, NO_SIGNAL.

In [1]:
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
REPO_ID = "FlyRank/internship-warehouse"

LANE = "content_refresh_prioritization"
MID_PANEL_MONTH = "2026-03"
PRIOR_MONTH = "2026-02"
DECISION_DATE = pd.Timestamp("2026-03-31")

def load_month(month):
    path = hf_hub_download(
        repo_id=REPO_ID, repo_type="dataset",
        filename=f"fact_content_daily_performance/month={month}/data_0.parquet",
        token=HF_TOKEN
    )
    return pd.read_parquet(path)

df_mid = load_month(MID_PANEL_MONTH)
df_prior = load_month(PRIOR_MONTH)

dim_content_path = hf_hub_download(repo_id=REPO_ID, repo_type="dataset", filename="dim_content.parquet", token=HF_TOKEN)
dim_content = pd.read_parquet(dim_content_path)

def aggregate_month(df, month):
    m = (
        df.groupby(["client_hash_id", "content_hash_id"], as_index=False)
        .agg(
            gsc_clicks=("gsc_clicks", "sum"),
            gsc_impressions=("gsc_impressions", "sum"),
            gsc_sum_position=("gsc_sum_position", "sum"),
            days_with_gsc=("gsc_data_available", "sum"),
        )
    )
    m["month"] = month
    # FIX: cast to float and use np.nan (not pd.NA) so ctr/avg_position stay numeric dtype
    impressions_f = m["gsc_impressions"].astype(float).replace(0, np.nan)
    m["ctr"] = m["gsc_clicks"].astype(float) / impressions_f
    m["avg_position"] = m["gsc_sum_position"].astype(float) / impressions_f
    return m[m["days_with_gsc"] > 0].copy()

monthly_mid = aggregate_month(df_mid, MID_PANEL_MONTH)
monthly_prior = aggregate_month(df_prior, PRIOR_MONTH)[["client_hash_id", "content_hash_id", "gsc_clicks"]].rename(columns={"gsc_clicks": "prior_clicks"})

base = monthly_mid.merge(monthly_prior, on=["client_hash_id", "content_hash_id"], how="left")
base = base.merge(
    dim_content[["client_hash_id", "content_hash_id", "content_created_date", "content_updated_date",
                 "last_optimized_date", "is_published", "is_deleted", "content_type"]],
    on=["client_hash_id", "content_hash_id"], how="left"
)

base["content_created_date"] = pd.to_datetime(base["content_created_date"], errors="coerce")
base["content_updated_date"] = pd.to_datetime(base["content_updated_date"], errors="coerce")

base = base[(base["is_published"] == True) & (base["is_deleted"] == False)]
base = base[base["content_created_date"].isna() | (base["content_created_date"] <= DECISION_DATE)]

base["trend_direction"] = base.apply(
    lambda r: "no_prior_data" if pd.isna(r["prior_clicks"])
    else ("down" if r["gsc_clicks"] < r["prior_clicks"] else ("up" if r["gsc_clicks"] > r["prior_clicks"] else "flat")),
    axis=1
)

print("Base table shape:", base.shape)
print(base[["gsc_clicks", "gsc_impressions", "ctr", "avg_position"]].describe())
print(base["trend_direction"].value_counts())

# SIGNAL 1: staleness (behind refresh flags)
base["days_since_update"] = (DECISION_DATE - base["content_updated_date"]).dt.days

def staleness_bucket(d):
    if pd.isna(d): return "unknown"
    if d <= 30: return "0-30d"
    if d <= 90: return "31-90d"
    if d <= 180: return "91-180d"
    if d <= 365: return "181-365d"
    return "365d+"

base["staleness_bucket"] = base["days_since_update"].apply(staleness_bucket)

signal1_table = (
    base.groupby("staleness_bucket")
    .agg(n=("content_hash_id", "count"), down_rate=("trend_direction", lambda s: (s == "down").mean()))
    .reindex(["0-30d", "31-90d", "91-180d", "181-365d", "365d+", "unknown"])
)
print("\n=== SIGNAL 1: staleness vs decline rate ===")
print(signal1_table)

# SIGNAL 2: CTR vs position (behind CTR-fix logic)
def position_bucket(p):
    if pd.isna(p): return "unknown"
    if p <= 3: return "1-3"
    if p <= 10: return "4-10"
    if p <= 20: return "11-20"
    return "21+"

base["position_bucket"] = base["avg_position"].apply(position_bucket)
bucket_median_ctr = base.groupby("position_bucket")["ctr"].transform("median")
base["ctr_underperform"] = base["ctr"] < (0.5 * bucket_median_ctr)

signal2_table = (
    base.groupby("position_bucket")
    .agg(n=("content_hash_id", "count"), median_ctr=("ctr", "median"), underperform_rate=("ctr_underperform", "mean"))
    .reindex(["1-3", "4-10", "11-20", "21+", "unknown"])
)
print("\n=== SIGNAL 2: CTR vs position ===")
print(signal2_table)

Base table shape: (176568, 17)
          gsc_clicks  gsc_impressions            ctr   avg_position
count  176568.000000    176568.000000  176568.000000  176568.000000
mean        4.653601      1589.294300       0.004597      15.999162
std        26.734474      5433.699984       0.037777      18.101554
min         0.000000         1.000000       0.000000       0.000000
25%         0.000000        20.000000       0.000000       4.918147
50%         0.000000       174.000000       0.000000       8.185845
75%         2.000000      1040.000000       0.002160      20.275399
max      5668.000000    617124.000000       1.000000     309.000000
trend_direction
flat             72364
no_prior_data    42482
up               37311
down             24411
Name: count, dtype: int64

=== SIGNAL 1: staleness vs decline rate ===
                         n  down_rate
staleness_bucket                     
0-30d             149439.0   0.128119
31-90d             25577.0   0.203190
91-180d             1324.0

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

STALE_CAP_DAYS = 180  # beyond this, extra staleness doesn't add more urgency

base["staleness_score"] = (base["days_since_update"].clip(upper=STALE_CAP_DAYS) / STALE_CAP_DAYS).fillna(0)

bucket_median_ctr = base.groupby("position_bucket")["ctr"].transform("median")
base["ctr_gap_score"] = (1 - (base["ctr"] / bucket_median_ctr)).clip(lower=0, upper=1).fillna(0)

base["action_score"] = (0.5 * base["staleness_score"] + 0.5 * base["ctr_gap_score"]).round(4)

def reason_code(row):
    if row["staleness_score"] >= 0.5 and row["ctr_gap_score"] >= 0.5:
        return "STALE_AND_CTR_GAP"
    if row["staleness_score"] > row["ctr_gap_score"]:
        return "STALE"
    if row["ctr_gap_score"] > row["staleness_score"]:
        return "CTR_GAP"
    return "NO_SIGNAL"

base["reason_code"] = base.apply(reason_code, axis=1)

def action_label(score):
    if score >= 0.66: return "refresh_now"
    if score >= 0.33: return "monitor"
    return "no_action"

base["action_label"] = base["action_score"].apply(action_label)

queue = base.sort_values("action_score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

output_cols = ["rank", "client_hash_id", "content_hash_id", "action_score", "reason_code", "action_label",
               "gsc_clicks", "gsc_impressions", "ctr", "avg_position", "days_since_update", "trend_direction"]

import os
os.makedirs("work/outputs", exist_ok=True)
queue[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Queue shape:", queue.shape)
print(queue["action_label"].value_counts())
queue[output_cols].head(20)


Queue shape: (176568, 27)
action_label
no_action    175217
monitor        1351
Name: count, dtype: int64


,rank,client_hash_id,content_hash_id,action_score,reason_code,action_label,gsc_clicks,gsc_impressions,ctr,avg_position,days_since_update,trend_direction
0,1,client_2b4306c3ed003f01,content_761e0b41690a9db2,0.5,STALE,monitor,0,14,0.000000,17.571429,191,flat
1,2,client_2b4306c3ed003f01,content_7616c3069520e530,0.5,STALE,monitor,0,3,0.000000,4.333333,188,no_prior_data
2,3,client_2b4306c3ed003f01,content_7b3b22dbfb08c1fc,0.5,STALE,monitor,0,11,0.000000,7.727273,191,flat
3,4,client_65de48885f4ef01b,content_a9e5a8f14112ddd3,0.5,STALE,monitor,2,482,0.004149,7.475104,234,up
4,5,client_73cda7b4e4f265ea,content_8cfdbd57ce4edd09,0.5,STALE,monitor,0,1,0.000000,8.000000,243,flat
5,6,client_c182d11e4862a37d,content_7dfe51e144f1c685,0.5,STALE,monitor,0,12,0.000000,6.333333,234,flat
6,7,client_2b4306c3ed003f01,content_7e654cfc57490b45,0.5,STALE,monitor,0,1,0.000000,10.000000,191,no_prior_data
7,8,client_73cda7b4e4f265ea,content_d6264d1edf1ff3cf,0.5,STALE,monitor,0,7,0.000000,15.000000,235,flat
8,9,client_2b4306c3ed003f01,content_82376476b51461f2,0.5,STALE,monitor,0,8,0.000000,12.875000,191,flat
9,10,client_2b4306c3ed003f01,content_80d737e7808fe68b,0.5,STALE,monitor,0,1,0.000000,9.000000,191,no_prior_data


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

content_hash_id=... — action: refresh_now, reason: STALE_AND_CTR_GAP (updated X days ago, CTR Y% below its position-bucket median). Would be wrong if: this page is intentionally evergreen and doesn't need edits, or if the CTR gap is because the query intent shifted, not the content quality.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
pd.set_option("display.max_colwidth", None)
top20 = queue[output_cols].head(20)
top20


,rank,client_hash_id,content_hash_id,action_score,reason_code,action_label,gsc_clicks,gsc_impressions,ctr,avg_position,days_since_update,trend_direction
0,1,client_2b4306c3ed003f01,content_761e0b41690a9db2,0.5,STALE,monitor,0,14,0.000000,17.571429,191,flat
1,2,client_2b4306c3ed003f01,content_7616c3069520e530,0.5,STALE,monitor,0,3,0.000000,4.333333,188,no_prior_data
2,3,client_2b4306c3ed003f01,content_7b3b22dbfb08c1fc,0.5,STALE,monitor,0,11,0.000000,7.727273,191,flat
3,4,client_65de48885f4ef01b,content_a9e5a8f14112ddd3,0.5,STALE,monitor,2,482,0.004149,7.475104,234,up
4,5,client_73cda7b4e4f265ea,content_8cfdbd57ce4edd09,0.5,STALE,monitor,0,1,0.000000,8.000000,243,flat
5,6,client_c182d11e4862a37d,content_7dfe51e144f1c685,0.5,STALE,monitor,0,12,0.000000,6.333333,234,flat
6,7,client_2b4306c3ed003f01,content_7e654cfc57490b45,0.5,STALE,monitor,0,1,0.000000,10.000000,191,no_prior_data
7,8,client_73cda7b4e4f265ea,content_d6264d1edf1ff3cf,0.5,STALE,monitor,0,7,0.000000,15.000000,235,flat
8,9,client_2b4306c3ed003f01,content_82376476b51461f2,0.5,STALE,monitor,0,8,0.000000,12.875000,191,flat
9,10,client_2b4306c3ed003f01,content_80d737e7808fe68b,0.5,STALE,monitor,0,1,0.000000,9.000000,191,no_prior_data


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage check
print("Inputs used to compute action_score: days_since_update, ctr, position_bucket (all knowable at decision moment).")
print("trend_direction was used ONLY in Section 1 to verify the staleness signal — it is NOT an input to action_score.")
assert "trend_direction" not in ["days_since_update", "ctr", "position_bucket"], "trend_direction accidentally used as a feature."

# Weak-looking picks near the bottom of the top-20
queue[output_cols].head(20).tail(5)


Inputs used to compute action_score: days_since_update, ctr, position_bucket (all knowable at decision moment).
trend_direction was used ONLY in Section 1 to verify the staleness signal — it is NOT an input to action_score.


,rank,client_hash_id,content_hash_id,action_score,reason_code,action_label,gsc_clicks,gsc_impressions,ctr,avg_position,days_since_update,trend_direction
15,16,client_73cda7b4e4f265ea,content_f4098d5b2c2eeb18,0.5,STALE,monitor,0,75,0.0,75.560000,243,flat
16,17,client_2b4306c3ed003f01,content_8d1bce995359d3b8,0.5,STALE,monitor,0,5,0.0,25.400000,191,flat
17,18,client_2b4306c3ed003f01,content_8c26f7eb45cf7b00,0.5,STALE,monitor,0,7,0.0,18.714286,191,no_prior_data
18,19,client_2b4306c3ed003f01,content_8bed4bb7415df0f7,0.5,STALE,monitor,0,33,0.0,28.000000,191,flat
19,20,client_73cda7b4e4f265ea,content_79ac27771747897b,0.5,STALE,monitor,0,13,0.0,2.153846,235,flat


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.